# 2048 AI - Validação no Google Colab

Este notebook valida se o Google Colab consegue clonar o projeto, instalar o pacote, importar o motor do jogo e avaliar os agentes de referência.

Nesta etapa ainda não vamos treinar uma rede neural. O objetivo é confirmar que a base técnica está pronta para o treinamento futuro.

## 1. Clonar ou atualizar o repositório

Execute esta célula primeiro. Ela baixa o código do GitHub para dentro do ambiente temporário do Colab.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/Sankofa-JBC/2048-ai.git"
PROJECT_DIR = Path("/content/2048-ai")

if PROJECT_DIR.exists():
    print("Repositorio ja existe. Atualizando...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=True)
else:
    print("Clonando repositorio...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Diretorio atual:", Path.cwd())

## 2. Instalar o projeto

O comando `pip install -e .` instala o pacote em modo editável. Isso permite importar `game2048` dentro do notebook.

In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

## 3. Validar importação do pacote

Se esta célula rodar sem erro, o Colab já consegue usar o motor do jogo e os agentes.

In [ ]:
from game2048 import Game2048, HeuristicAgent, RandomAgent

game = Game2048(seed=42)
print("Classe do jogo:", Game2048.__name__)
print("Agente aleatorio:", RandomAgent.name)
print("Agente heuristico:", HeuristicAgent.name)
print("Tabuleiro inicial:")
for row in game.board:
    print(row)

## 4. Rodar os testes

Os testes confirmam que as regras do jogo, os agentes e o avaliador continuam funcionando dentro do Colab.

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"], check=True)

## 5. Avaliar os agentes de referência

Aqui comparamos o agente aleatório com o agente heurístico usando a mesma seed base.

In [ ]:
subprocess.run([sys.executable, "evaluate_agents.py", "--agent", "random", "--games", "100", "--seed", "42"], check=True)
subprocess.run([sys.executable, "evaluate_agents.py", "--agent", "heuristic", "--games", "100", "--seed", "42"], check=True)

## 6. Exportar métricas

Os arquivos gerados em `results/` serão úteis para comparar experimentos depois.

In [ ]:
from pathlib import Path

Path("results").mkdir(exist_ok=True)

subprocess.run([
    sys.executable,
    "evaluate_agents.py",
    "--agent",
    "random",
    "--games",
    "100",
    "--seed",
    "42",
    "--output",
    "results/random_colab.json",
], check=True)

subprocess.run([
    sys.executable,
    "evaluate_agents.py",
    "--agent",
    "heuristic",
    "--games",
    "100",
    "--seed",
    "42",
    "--output",
    "results/heuristic_colab.json",
], check=True)

## 7. Ler os resultados exportados

Esta célula carrega os JSONs e mostra um resumo simples dentro do notebook.

In [ ]:
import json

for file_name in ["results/random_colab.json", "results/heuristic_colab.json"]:
    with open(file_name, "r", encoding="utf-8") as file:
        data = json.load(file)

    summary = data["summary"]
    print(file_name)
    print("  agente:", summary["agent_name"])
    print("  jogos:", summary["games"])
    print("  score medio:", round(summary["average_score"], 2))
    print("  melhor score:", summary["best_score"])
    print("  maior bloco:", summary["best_max_tile"])
    print()

## Próximo passo

Se todas as células rodarem sem erro, o ambiente Colab está validado. Depois disso podemos criar o primeiro algoritmo de aprendizagem.